This is version 2 of my Fantasy Hockey Analyzer. The purpose of this notebook is to predict the number of fantasy points every hockey player in the league will get based on previous years' performance.

This notebook primarily uses data from moneypuck.com for analysis, and it also uses data from rotowire.com to get +/- for each player.

Section 1: Parameters and Modules

These are the variables that can be adjusted. My model is an ensemble model consisting of neural nets, random forests, and boosted trees, with data going back one, two, and three years.

In [1]:
# Set these values to the appropriate ammonts

current_year = 2026
common_number = 30
number_of_one_year_neural_nets = common_number
number_of_two_year_neural_nets = common_number
number_of_three_year_neural_nets = common_number
number_of_one_year_random_forests = common_number
number_of_two_year_random_forests = common_number
number_of_three_year_random_forests = common_number
number_of_one_year_boosted_trees = common_number
number_of_two_year_boosted_trees = common_number
number_of_three_year_boosted_trees = common_number

# Run training when True; use saved models without training when False.
create_new_models = True

# Reuse completed models from a matching run. False starts each trained group fresh.
# After an interruption, keep this True to preserve completed training.
resume_training = True

# This is the breakdown of how many fantasy points a player gets for each category
points_dictionary = {
    'Goals':5, 
    'Assists':3, 
    '+/-':1.5, 
    'PIM':-0.25, 
    'PP_Goals':4, 
    'PP_Assists':2, 
    'SH_Goals':6, #won't count SHG from 5-on-3
    'SH_Assists':4, 
    'Faceoffs_Won':0.25, 
    'Faceoffs_Lost':-0.15, 
    'Hits':0.5, 
    'Blocked_Shots':0.75
    }



The following is a list of modules that I used and the reason why they were used:

-os: to allow the program to read data in the repository

-numpy: basic math operations

-pandas: all dataframe operations/data storage/data cleaning

-various sklearn: all machine learning operations/analysis

In addition to these modules, I also have a custom module that contains helper functions that help in data cleaning/accuracy evaluation. These functions are contained in the "my_module_v2.py" file in the repository. If you are interested in taking a look at these functions, they are available at https://github.com/chrisberry888/FantasyHockeyAnalyzer in the "my_module_v2.py" file.

In [2]:
# Change kernel's working directory to the project root:
%cd ..

#Import block
import os
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.base import clone
import joblib
import my_module_v2 as mx

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None) 

/home/chris/Git_Repositories/FantasyHockeyAnalyzer


Section 2: Data Gathering and Cleaning

This section compiles the Moneypuck and Rotowire data into a format that is usable by the ML models.

There is some discrepencies between stats on ESPN and on moneypuck. These shouldn't alter the fantasy points too much. For example, Sydney Crosby is short-changed 1 faceoff win but has one additional hit in moneypuck than on ESPN, so in my model he has 0.10 more points than he does on ESPN. I would like to see why this is the case in the future/get fully accurate data (perhaps from ESPN themselves), but for now I am ok with this very small error.

In [3]:
yearly_player_data = []
unmatched_player_tables = []

for year in range(2010, current_year):
    moneypuck_data = mx.get_moneypuck_data(year)
    rotowire_data = mx.get_rotowire_data(year)
    combined_data, unmatched_players = mx.combine_dataframes(
        moneypuck_data, rotowire_data, return_unmatched=True
    )
    unmatched_player_tables.append(unmatched_players)
    this_years_data = mx.calculate_fantasy_points(combined_data, points_dictionary)
    yearly_player_data.append(this_years_data)

# Record unresolved player identities across every loaded season.
pd.concat(unmatched_player_tables, ignore_index=True).to_csv(
    'unmatched_players.csv', index=False
)


/home/chris/Git_Repositories/FantasyHockeyAnalyzer/my_module_v2.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df = pivoted_df.reset_index()
/home/chris/Git_Repositories/FantasyHockeyAnalyzer/my_module_v2.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df = pivoted_df.reset_index()
/home/chris/Git_Repositories/FantasyHockeyAnalyzer/my_module_v2.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

In [4]:
player_id_table = mx.get_player_id_table(yearly_player_data)

This next cell compiles the yearly data into chunks of one, two, and three-year data to be used by the ML models.

In [5]:
ml_data_one_year = mx.get_ml_data(yearly_player_data, current_year, 1)
ml_data_two_years = mx.get_ml_data(yearly_player_data, current_year, 2)
ml_data_three_years = mx.get_ml_data(yearly_player_data, current_year, 3)

The data is now ready to be used to train the ML model.

In [6]:

one_year_X, one_year_y = mx.separate_fantasy_points(ml_data_one_year)
two_year_X, two_year_y = mx.separate_fantasy_points(ml_data_two_years)
three_year_X, three_year_y = mx.separate_fantasy_points(ml_data_three_years)

In [7]:
one_year_neural_net_args = (
    one_year_X,
    one_year_y,
    make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(50,),
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
        ),
    ),
    number_of_one_year_neural_nets,
	'1_year',
	'neural_nets'
)

one_year_random_forest_args = (
    one_year_X,
    one_year_y,
    RandomForestRegressor(
        n_estimators=50,
        max_features=0.2,
        max_depth=10,
        min_samples_leaf=5,
        n_jobs=-1,
    ),
    number_of_one_year_random_forests,
	'1_year',
	'random_forests'
)

one_year_boosted_tree_args = (
    one_year_X,
    one_year_y,
    HistGradientBoostingRegressor(
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
    ),
    number_of_one_year_boosted_trees,
	'1_year',
	'boosted_trees'
)

two_year_neural_net_args = (
    two_year_X,
    two_year_y,
    make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(50,),
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
        ),
    ),
    number_of_two_year_neural_nets,
	'2_year',
	'neural_nets'
)

two_year_random_forest_args = (
    two_year_X,
    two_year_y,
    RandomForestRegressor(
        n_estimators=50,
        max_features=0.2,
        max_depth=10,
        min_samples_leaf=5,
        n_jobs=-1,
    ),
    number_of_two_year_random_forests,
	'2_year',
	'random_forests'
)

two_year_boosted_tree_args = (
    two_year_X,
    two_year_y,
    HistGradientBoostingRegressor(
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
    ),
    number_of_two_year_boosted_trees,
	'2_year',
	'boosted_trees'
)

three_year_neural_net_args = (
    three_year_X,
    three_year_y,
    make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(50,),
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
        ),
    ),
    number_of_three_year_neural_nets,
	'3_year',
	'neural_nets'
)

three_year_random_forest_args = (
    three_year_X,
    three_year_y,
    RandomForestRegressor(
        n_estimators=50,
        max_features=0.2,
        max_depth=10,
        min_samples_leaf=5,
        n_jobs=-1,
    ),
    number_of_three_year_random_forests,
	'3_year',
	'random_forests'
)

three_year_boosted_tree_args = (
    three_year_X,
    three_year_y,
    HistGradientBoostingRegressor(
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
    ),
    number_of_three_year_boosted_trees,
	'3_year',
	'boosted_trees'
)

In [8]:

if create_new_models:
    mx.create_models(*one_year_neural_net_args, resume_training=resume_training)
    mx.create_models(*one_year_random_forest_args, resume_training=resume_training)
    mx.create_models(*one_year_boosted_tree_args, resume_training=resume_training)
    mx.create_models(*two_year_neural_net_args, resume_training=resume_training)
    mx.create_models(*two_year_random_forest_args, resume_training=resume_training)
    mx.create_models(*two_year_boosted_tree_args, resume_training=resume_training)
    mx.create_models(*three_year_neural_net_args, resume_training=resume_training)
    mx.create_models(*three_year_random_forest_args, resume_training=resume_training)
    mx.create_models(*three_year_boosted_tree_args, resume_training=resume_training)
    


1_year/neural_nets: saved model 1/30
1_year/neural_nets: saved model 2/30
1_year/neural_nets: saved model 3/30
1_year/neural_nets: saved model 4/30
1_year/neural_nets: saved model 5/30
1_year/neural_nets: saved model 6/30
1_year/neural_nets: saved model 7/30
1_year/neural_nets: saved model 8/30
1_year/neural_nets: saved model 9/30
1_year/neural_nets: saved model 10/30
1_year/neural_nets: saved model 11/30
1_year/neural_nets: saved model 12/30
1_year/neural_nets: saved model 13/30
1_year/neural_nets: saved model 14/30
1_year/neural_nets: saved model 15/30
1_year/neural_nets: saved model 16/30
1_year/neural_nets: saved model 17/30
1_year/neural_nets: saved model 18/30
1_year/neural_nets: saved model 19/30
1_year/neural_nets: saved model 20/30
1_year/neural_nets: saved model 21/30
1_year/neural_nets: saved model 22/30
1_year/neural_nets: saved model 23/30
1_year/neural_nets: saved model 24/30
1_year/neural_nets: saved model 25/30
1_year/neural_nets: saved model 26/30
1_year/neural_nets: s

Now we generate the table with final predictions.

In [9]:
current_one_year_X = mx.get_final_year_data(yearly_player_data, 1)
current_two_year_X = mx.get_final_year_data(yearly_player_data, 2)
current_three_year_X = mx.get_final_year_data(yearly_player_data, 3)

In [10]:
path = os.getcwd() + '/models/1_year/neural_nets/model_0.joblib'
model = joblib.load(path)
table = mx.get_prediction_table([model], current_one_year_X, player_id_table)

In [11]:
final_table_inputs = (
    (
        current_one_year_X,
        current_two_year_X,
        current_three_year_X
    ),
    player_id_table
)
mx.generate_predictions(*final_table_inputs)

In [12]:
mx.generate_final_table()

In [13]:
mx.get_final_table()

,playerID,Player Name,Team,Position,Prediction,model_coverage_percent
0,8478402,Connor McDavid,EDM,C,663.852139,100.0
1,8477492,Nathan MacKinnon,COL,C,625.042354,100.0
2,8477934,Leon Draisaitl,EDM,C,610.689766,100.0
3,8484801,Macklin Celebrini,SJS,C,571.796546,66.7
4,8480018,Nick Suzuki,MTL,C,545.981146,100.0
5,8478403,Jack Eichel,VGK,C,538.279120,100.0
6,8480002,Nico Hischier,NJD,C,504.609448,100.0
7,8476453,Nikita Kucherov,TBL,R,496.405480,100.0
8,8476460,Mark Scheifele,WPG,C,493.927032,100.0
9,8476881,Tomas Hertl,VGK,C,473.286020,100.0


In [14]:
mx.get_nhl_players().head()

,player_id,team_abbrev,position
0,8484153,ANA,C
1,8481538,ANA,R
2,8482118,ANA,R
3,8482081,ANA,C
4,8483444,ANA,C
